In [0]:
# === SILVER LAYER: Production-Grade Data Profiling & Cleaning Pipeline ===
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, row_number, when, round as spark_round, mean, trim, initcap,
    to_date, to_timestamp, coalesce, count, lit, sum as spark_sum,
    min as spark_min, max as spark_max, avg, countDistinct, isnan, upper,
    desc, abs as spark_abs
)
from pyspark.sql.types import DoubleType, IntegerType, StringType
from functools import reduce

# ── Load Bronze Tables ────────────────────────────────────────────────────────
datasets = {}
datasets["sales"]       = spark.table("medallion.bronze.sales_transactions")
datasets["product"]     = spark.table("medallion.bronze.product_master")
datasets["customer"]    = spark.table("medallion.bronze.customer_data")
datasets["store"]       = spark.table("medallion.bronze.store_master")
datasets["inventory"]   = spark.table("medallion.bronze.inventory_data")
datasets["clickstream"] = spark.table("medallion.bronze.clickstream_events")

# Convenience aliases
sales_df, product_df = datasets["sales"], datasets["product"]
customer_df, store_df = datasets["customer"], datasets["store"]
inventory_df, clickstream_df = datasets["inventory"], datasets["clickstream"]

# ── Capture PRE-cleaning baselines (for Step 4 comparison) ────────────────────
pre_row_counts = {name: df.count() for name, df in datasets.items()}

def null_profile(df):
    """Return {col_name: null_count} for every column."""
    return {c: df.filter(col(c).isNull()).count() for c in df.columns}

pre_nulls = {name: null_profile(df) for name, df in datasets.items()}

def dup_count(df, key_col):
    return df.count() - df.dropDuplicates([key_col]).count()

pre_duplicates = {
    "sales":       dup_count(sales_df, "transaction_id"),
    "product":     dup_count(product_df, "product_id"),
    "customer":    dup_count(customer_df, "customer_id"),
    "store":       dup_count(store_df, "store_id"),
    "inventory":   inventory_df.count() - inventory_df.dropDuplicates(["product_id", "store_id"]).count(),
    "clickstream": dup_count(clickstream_df, "event_id")
}

print("✅ Bronze tables loaded  |  Pre-cleaning baselines captured")

✅ Bronze tables loaded  |  Pre-cleaning baselines captured


# 📊 STEP 1: Initial EDA — Before Cleaning
> Profile every Bronze dataset to document data quality issues **before** any transformations are applied.

In [0]:
# ── 1a. Schema, Sample Records & Row Counts ──────────────────────────────────
print("\n" + "=" * 70)
print("   ROW COUNTS (Bronze Layer)")
print("=" * 70)
for name, cnt in pre_row_counts.items():
    print(f"   {name:<15} {cnt:>10,} rows")

# Schema + sample for each dataset
for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"   SCHEMA: {name.upper()}")
    print(f"{'=' * 70}")
    df.printSchema()
    print(f"   Sample records ({name}):")
    display(df.limit(5))


   ROW COUNTS (Bronze Layer)
   sales            1,000,500 rows
   product                500 rows
   customer           100,000 rows
   store                  100 rows
   inventory            2,500 rows
   clickstream      1,500,000 rows

   SCHEMA: SALES
root
 |-- transaction_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- ingestion_timestamp: date (nullable = true)

   Sample records (sales):


transaction_id,order_date,channel,store_id,product_id,customer_id,quantity,unit_price,discount,total_amount,payment_type,ingestion_timestamp
5009060,2024-03-23,Store,30,1027,107714,1,704.2830668255222,0.17,584.55,Credit Card,2024-03-23
5009203,2024-03-23,Store,97,1111,102012,5,465.00439611687773,0.26,1720.52,Credit Card,2024-03-24
5010177,2024-03-23,Store,61,1007,65803,5,333.3901572687763,0.07,1550.26,Debit Card,2024-03-23
5012081,2024-03-23,Store,59,1046,105395,2,913.6890476934449,0.2,1461.9,Cash,2024-03-24
5013143,2024-03-23,Store,10,1178,21693,3,795.8292432385986,0.18,1957.74,Debit Card,2024-03-23



   SCHEMA: PRODUCT
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- selling_price: double (nullable = true)

   Sample records (product):


product_id,product_name,category,sub_category,brand,cost_price,selling_price
1000,Product_0,Sports,Cosmetics,Adidas,527.92,616.8195044634394
1001,Product_1,Beauty,Furniture,Sony,785.43,938.9047330115245
1002,Product_2,Home,Mobile,Samsung,727.52,857.9422906691381
1003,Product_3,Beauty,Cosmetics,Apple,535.0,708.8359867605269
1004,Product_4,Beauty,Fitness,Sony,569.88,799.0052005180304



   SCHEMA: CUSTOMER
root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- loyalty_status: string (nullable = true)
 |-- signup_channel: string (nullable = true)
 |-- city: string (nullable = true)

   Sample records (customer):


...

store_id,store_name,city,state,store_type
1,Store_0,Mumbai,Telangana,Warehouse
2,Store_1,Mumbai,Punjab,Mall
3,Store_2,Bangalore,Telangana,Mall
4,Store_3,Delhi,Rajasthan,Mall
5,Store_4,Pune,Telangana,Standalone



   SCHEMA: INVENTORY
root
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_updated: timestamp (nullable = true)

   Sample records (inventory):


product_id,store_id,stock_on_hand,reorder_level,last_updated
1000,2,42,20,2026-03-29T07:03:39.795824Z
1000,61,0,38,2026-03-29T07:03:39.795832Z
1000,11,324,96,2026-03-29T07:03:39.795836Z
1000,48,497,26,2026-03-29T07:03:39.795839Z
1000,69,263,63,2026-03-29T07:03:39.795842Z



   SCHEMA: CLICKSTREAM
root
 |-- event_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: date (nullable = true)
 |-- platform: string (nullable = true)

   Sample records (clickstream):


event_id,customer_id,product_id,event_type,event_timestamp,platform
8757951,82840,1107,purchase,2024-08-28,Mobile
8757952,55040,1068,view,2024-09-24,Web
8757953,106367,1148,view,2024-03-23,Mobile
8757954,71991,1283,add_to_cart,2024-08-06,Mobile
8757955,13461,1040,view,2024-07-05,Web


In [0]:
# ── 1b. NULL Analysis (all columns, all datasets) ──────────────────────────
from pyspark.sql import Row

for name, null_map in pre_nulls.items():
    total = pre_row_counts[name]
    print(f"\n{'=' * 70}")
    print(f"   NULL ANALYSIS: {name.upper()}  ({total:,} rows)")
    print(f"{'=' * 70}")
    rows = [
        Row(column=c, null_count=n, pct_null=round(n / total * 100, 2))
        for c, n in null_map.items() if n > 0
    ]
    if rows:
        display(spark.createDataFrame(rows))
    else:
        print("   ✅ No nulls found.")


   NULL ANALYSIS: SALES  (1,000,500 rows)
   ✅ No nulls found.

   NULL ANALYSIS: PRODUCT  (500 rows)
   ✅ No nulls found.

   NULL ANALYSIS: CUSTOMER  (100,000 rows)


column,null_count,pct_null
age,200,0.2



   NULL ANALYSIS: STORE  (100 rows)
   ✅ No nulls found.

   NULL ANALYSIS: INVENTORY  (2,500 rows)
   ✅ No nulls found.

   NULL ANALYSIS: CLICKSTREAM  (1,500,000 rows)
   ✅ No nulls found.


In [0]:
# ── 1c. Duplicate Records ─────────────────────────────────────────────────
print("=" * 70)
print("   DUPLICATE RECORDS (Bronze Layer)")
print("=" * 70)
for name, cnt in pre_duplicates.items():
    flag = "⚠️" if cnt > 0 else "✅"
    print(f"   {flag} {name:<15} {cnt:>6} duplicates")

# ── 1d. Data Quality Issues (business-rule violations) ──────────────────
print(f"\n{'=' * 70}")
print("   DATA QUALITY ISSUES")
print(f"{'=' * 70}")

# Sales issues
qty_bad   = sales_df.filter(col("quantity") <= 0).count()
price_bad = sales_df.filter(col("unit_price") <= 0).count()
total_bad = sales_df.filter(col("total_amount") < 0).count()
print(f"\n   💰 SALES:")
print(f"      quantity <= 0:      {qty_bad}")
print(f"      unit_price <= 0:    {price_bad}")
print(f"      total_amount < 0:   {total_bad}")

# Product issues
loss_products = product_df.filter(col("selling_price") < col("cost_price")).count()
print(f"\n   📦 PRODUCT:")
print(f"      selling_price < cost_price: {loss_products}")

# Inventory issues
neg_stock = inventory_df.filter(col("stock_on_hand") < 0).count()
print(f"\n   📦 INVENTORY:")
print(f"      stock_on_hand < 0:  {neg_stock}")

# Store pre_quality for later comparison
pre_quality = {
    "sales_qty_bad": qty_bad, "sales_price_bad": price_bad,
    "sales_total_bad": total_bad, "product_loss": loss_products,
    "inv_neg_stock": neg_stock
}

   DUPLICATE RECORDS (Bronze Layer)
   ⚠️ sales              500 duplicates
   ✅ product              0 duplicates
   ✅ customer             0 duplicates
   ✅ store                0 duplicates
   ✅ inventory            0 duplicates
   ✅ clickstream          0 duplicates

   DATA QUALITY ISSUES

   💰 SALES:
      quantity <= 0:      50
      unit_price <= 0:    50
      total_amount < 0:   0

   📦 PRODUCT:
      selling_price < cost_price: 10

   📦 INVENTORY:
      stock_on_hand < 0:  20


In [0]:
# ── 1e. Distinct Categorical Values & Inconsistencies ───────────────────
print("=" * 70)
print("   CATEGORICAL COLUMN AUDIT")
print("=" * 70)

categorical_checks = [
    ("product",   product_df,   ["category", "brand"]),
    ("store",     store_df,     ["city", "state"]),
    ("customer",  customer_df,  ["gender", "loyalty_status"]),
    ("sales",     sales_df,     ["channel", "payment_type"]),
]

pre_distinct = {}
for ds_name, df, cols in categorical_checks:
    for c in cols:
        vals = [row[c] for row in df.select(c).distinct().collect()]
        pre_distinct[f"{ds_name}.{c}"] = sorted([str(v) for v in vals if v])
        print(f"\n   {ds_name}.{c} ({len(vals)} distinct):")
        print(f"      {vals[:20]}")

# Flag inconsistencies
print(f"\n{'=' * 70}")
print("   ⚠️  KNOWN INCONSISTENCIES")
print(f"{'=' * 70}")
print("   product.category:  Look for 'Electroncs', 'Eletronics', 'Clths'")
print("   store.state:       Look for 'Tn', 'Tamilnadu', 'Karnataka '")
print("   customer.gender:   Look for mixed case (Male/MALE/male)")

# ── 1f. Summary Statistics (numeric columns) ───────────────────────────
print(f"\n{'=' * 70}")
print("   SUMMARY STATISTICS")
print(f"{'=' * 70}")

stats_checks = [
    ("sales",     sales_df,     ["quantity", "unit_price", "discount", "total_amount"]),
    ("product",   product_df,   ["cost_price", "selling_price"]),
    ("customer",  customer_df,  ["age"]),
    ("inventory", inventory_df, ["stock_on_hand", "reorder_level"]),
]

for ds_name, df, cols in stats_checks:
    agg_exprs = []
    for c in cols:
        agg_exprs += [
            spark_round(avg(c), 2).alias(f"{c}_mean"),
            spark_min(c).alias(f"{c}_min"),
            spark_max(c).alias(f"{c}_max")
        ]
    print(f"\n   {ds_name.upper()}:")
    display(df.agg(*agg_exprs))

   CATEGORICAL COLUMN AUDIT

   product.category (5 distinct):
      ['Home', 'Sports', 'Electronics', 'Clothing', 'Beauty']

   product.brand (7 distinct):
      ['Nike', 'Sony', 'Puma', 'Samsung', 'LG', 'Apple', 'Adidas']

   store.city (7 distinct):
      ['Bangalore', 'Mumbai', 'Pune', 'Delhi', 'Chandigarh', 'Hyderabad', 'Jaipur']

   store.state (6 distinct):
      ['Karnataka', 'Punjab', 'Delhi', 'Rajasthan', 'Telangana', 'Maharashtra']

   customer.gender (2 distinct):
      ['Female', 'Male']

   customer.loyalty_status (4 distinct):
      ['None', 'Silver', 'Gold', 'Bronze']

   sales.channel (3 distinct):
      ['Store', 'Online', 'Mobile']

   sales.payment_type (4 distinct):
      ['Credit Card', 'Cash', 'Debit Card', 'UPI']

   ⚠️  KNOWN INCONSISTENCIES
   product.category:  Look for 'Electroncs', 'Eletronics', 'Clths'
   store.state:       Look for 'Tn', 'Tamilnadu', 'Karnataka '
   customer.gender:   Look for mixed case (Male/MALE/male)

   SUMMARY STATISTICS

   SALES:


quantity_mean,quantity_min,quantity_max,unit_price_mean,unit_price_min,unit_price_max,discount_mean,discount_min,discount_max,total_amount_mean,total_amount_min,total_amount_max
3.5,-1,8,575.17,0.0,1089.3010802094873,0.15,0.0,0.3,1466.68,28.13,5446.51



   PRODUCT:


cost_price_mean,cost_price_min,cost_price_max,selling_price_mean,selling_price_min,selling_price_max
428.8,50.18,799.52,581.83,40.18,1089.3010802094873



   CUSTOMER:


age_mean,age_min,age_max
43.46,18.0,69.0



   INVENTORY:


stock_on_hand_mean,stock_on_hand_min,stock_on_hand_max,reorder_level_mean,reorder_level_min,reorder_level_max
245.64,-5,500,54.58,10,100


# 🛠️ STEP 2: Data Cleaning & Transformation
> Apply retail business rules: deduplication, quality filters, master data mappings, financial enrichment.

In [0]:
# ═══ CUSTOMER — One-Pass Cleaning ═════════════════════════════════════════════════
dynamic_avg_age = int(
    customer_df.select(spark_round(mean(col("age")), 0)).first()[0]
)
print(f"Dynamic mean customer age: {dynamic_avg_age}")

silver_customer = customer_df \
    .withColumn("gender", upper(trim(col("gender")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn(
        "loyalty_status",
        when(
            col("loyalty_status").isNull() | (col("loyalty_status") == "None"),
            "Standard"
        ).otherwise(col("loyalty_status"))
    ) \
    .withColumn("age", when(col("age").isNull(), lit(dynamic_avg_age)).otherwise(col("age"))) \
    .fillna({"city": "Unknown"})

# ═══ INVENTORY — Ghost Inventory Fix ══════════════════════════════════════════════
# Business rule: negative stock = system sync error → floor at 0
silver_inventory = inventory_df \
    .withColumn("stock_on_hand",
        when(col("stock_on_hand") < 0, 0).otherwise(col("stock_on_hand"))) \
    .withColumn("last_updated", to_timestamp(col("last_updated")))

# ═══ STORE MASTER — Geography Mapping ════════════════════════════════════════════
geo_mapping = [
    ("Mumbai", "Maharashtra"), ("Pune", "Maharashtra"),
    ("Bangalore", "Karnataka"), ("Delhi", "Delhi"),
    ("Hyderabad", "Telangana"), ("Chandigarh", "Chandigarh"),
    ("Jaipur", "Rajasthan")
]
geo_map_df = spark.createDataFrame(geo_mapping, ["city", "correct_state"])

silver_store = store_df \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", initcap(trim(col("state")))) \
    .withColumn("store_type", initcap(trim(col("store_type")))) \
    .withColumn("state",
        when(col("state").isin("Tn", "Tamilnadu"), "Tamil Nadu").otherwise(col("state"))) \
    .join(geo_map_df, on="city", how="left") \
    .withColumn("state", coalesce(col("correct_state"), col("state"))) \
    .drop("correct_state") \
    .fillna({"city": "Unknown", "state": "Unknown"})

# ═══ PRODUCT MASTER — Brand Hierarchy Correction ═════════════════════════════════
brand_mapping = [
    ("Apple", "Electronics", "Mobile"), ("Samsung", "Electronics", "Mobile"),
    ("Sony", "Electronics", "Audio/Video"), ("LG", "Electronics", "Appliances"),
    ("Nike", "Clothing/Sports", "Shoes"), ("Adidas", "Clothing/Sports", "Shoes"),
    ("Puma", "Clothing/Sports", "Shoes")
]
brand_map_df = spark.createDataFrame(brand_mapping, ["brand", "correct_category", "correct_sub_category"])

# Standardize first, then apply brand hierarchy
silver_product = product_df \
    .withColumn("brand", upper(trim(col("brand")))) \
    .withColumn("category", initcap(trim(col("category")))) \
    .withColumn("sub_category", initcap(trim(col("sub_category")))) \
    .withColumn("category",
        when(col("category").isin("Electroncs", "Eletronics"), "Electronics")
        .when(col("category") == "Clths", "Clothing")
        .otherwise(col("category"))) \
    .join(brand_map_df, on="brand", how="left") \
    .withColumn("category", coalesce(col("correct_category"), col("category"))) \
    .withColumn("sub_category", coalesce(col("correct_sub_category"), col("sub_category"))) \
    .drop("correct_category", "correct_sub_category")

print("✅ Dimension tables cleaned (Customer, Inventory, Store, Product)")

Dynamic mean customer age: 43
✅ Dimension tables cleaned (Customer, Inventory, Store, Product)


In [0]:
# ═══ SALES — Deduplication ═════════════════════════════════════════════════════
window_spec = Window.partitionBy("transaction_id").orderBy(col("ingestion_timestamp").desc())

deduped_sales = sales_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# ═══ SALES — Quality Filter & Reject Table ═════════════════════════════════════
valid_condition = (col("quantity") > 0) & (col("unit_price") > 0) & (col("total_amount") >= 0)

clean_sales  = deduped_sales.filter(valid_condition)
reject_sales = deduped_sales.filter(~valid_condition)

print(f"Valid sales:    {clean_sales.count():,}")
print(f"Rejected sales: {reject_sales.count():,}")

# ═══ SALES — Type Standardization & Recalculation ═════════════════════════════
clean_sales = clean_sales \
    .withColumn("order_date", to_date(col("order_date"))) \
    .withColumn("calculated_total", spark_round(col("quantity") * col("unit_price") - col("discount"), 2)) \
    .withColumn("total_amount",
        when(spark_abs(col("total_amount") - col("calculated_total")) > 1,
             col("calculated_total"))
        .otherwise(col("total_amount"))) \
    .drop("calculated_total")

# ═══ SALES — Financial Enrichment ═════════════════════════════════════════════
silver_sales = clean_sales \
    .withColumn("net_sales",
        spark_round(col("quantity") * col("unit_price") * (1 - col("discount")), 2)) \
    .withColumn("Order_Value_Band",
        when(col("total_amount") >= 2000, "High")
        .when(col("total_amount") >= 500, "Medium")
        .otherwise("Low"))

# Join with product for gross_margin = net_sales - (quantity * cost_price)
silver_sales = silver_sales.alias("s") \
    .join(silver_product.alias("p"), col("s.product_id") == col("p.product_id"), "left") \
    .withColumn("gross_margin",
        spark_round(col("s.net_sales") - (col("s.quantity") * col("p.cost_price")), 2)) \
    .select("s.*", "gross_margin")

# ═══ CLICKSTREAM — Standardize & Deduplicate ═══════════════════════════════════
silver_clickstream = clickstream_df \
    .withColumn("event_timestamp", to_timestamp(col("event_timestamp"))) \
    .dropDuplicates()

# Collect all silver DataFrames
silver_datasets = {
    "sales": silver_sales, "product": silver_product,
    "customer": silver_customer, "store": silver_store,
    "inventory": silver_inventory, "clickstream": silver_clickstream
}

print("✅ Sales pipeline, clickstream standardization & enrichment complete.")

Valid sales:    999,900
Rejected sales: 100
✅ Sales pipeline, clickstream standardization & enrichment complete.


In [0]:
# ──────────────────────────────────────────────────────────────────────
# STEP 3: POST-CLEANING EDA
# ──────────────────────────────────────────────────────────────────────

# ── 3a. Null Analysis (post-cleaning) ───────────────────────────────────
post_nulls = {name: null_profile(df) for name, df in silver_datasets.items()}

print("=" * 70)
print("   POST-CLEANING: NULL ANALYSIS")
print("=" * 70)
for name, null_map in post_nulls.items():
    remaining = sum(null_map.values())
    print(f"   {name:<15} {remaining:>6} total nulls remaining")

# ── 3b. Duplicate Check (post-cleaning) ─────────────────────────────────
print(f"\n{'=' * 70}")
print("   POST-CLEANING: DUPLICATE CHECK")
print("=" * 70)
post_dups = {
    "sales":       dup_count(silver_sales, "transaction_id"),
    "product":     dup_count(silver_product, "product_id"),
    "customer":    dup_count(silver_customer, "customer_id"),
    "store":       dup_count(silver_store, "store_id"),
    "inventory":   silver_inventory.count() - silver_inventory.dropDuplicates(["product_id", "store_id"]).count(),
    "clickstream": dup_count(silver_clickstream, "event_id")
}
for name, cnt in post_dups.items():
    flag = "✅" if cnt == 0 else "⚠️"
    print(f"   {flag} {name:<15} {cnt:>6} duplicates")

# ── 3c. Business Rule Validation (post-cleaning) ───────────────────────
print(f"\n{'=' * 70}")
print("   POST-CLEANING: BUSINESS RULE VALIDATION")
print("=" * 70)
post_quality = {
    "sales_qty_bad":   silver_sales.filter(col("quantity") <= 0).count(),
    "sales_price_bad": silver_sales.filter(col("unit_price") <= 0).count(),
    "sales_total_bad": silver_sales.filter(col("total_amount") < 0).count(),
    "product_loss":    silver_product.filter(col("selling_price") < col("cost_price")).count(),
    "inv_neg_stock":   silver_inventory.filter(col("stock_on_hand") < 0).count()
}
for rule, cnt in post_quality.items():
    flag = "✅" if cnt == 0 else "⚠️"
    print(f"   {flag} {rule:<25} {cnt:>6} violations")

# ── 3d. Categorical Audit (post-cleaning) ───────────────────────────────
print(f"\n{'=' * 70}")
print("   POST-CLEANING: CATEGORICAL VALUES")
print("=" * 70)
post_categorical = [
    ("product",  silver_product,  ["category", "brand"]),
    ("store",    silver_store,    ["city", "state"]),
    ("customer", silver_customer, ["gender", "loyalty_status"]),
    ("sales",    silver_sales,    ["channel", "Order_Value_Band"]),
]
for ds_name, df, cols in post_categorical:
    for c in cols:
        vals = sorted([str(row[c]) for row in df.select(c).distinct().collect() if row[c]])
        print(f"   {ds_name}.{c}: {vals}")

# ── 3e. Summary Statistics (post-cleaning) ─────────────────────────────
print(f"\n{'=' * 70}")
print("   POST-CLEANING: SUMMARY STATISTICS")
print("=" * 70)
for ds_name, df, cols in [
    ("sales", silver_sales, ["quantity", "unit_price", "discount", "net_sales", "gross_margin"]),
    ("product", silver_product, ["cost_price", "selling_price"]),
    ("customer", silver_customer, ["age"]),
    ("inventory", silver_inventory, ["stock_on_hand"])
]:
    agg_exprs = []
    for c in cols:
        agg_exprs += [
            spark_round(avg(c), 2).alias(f"{c}_mean"),
            spark_min(c).alias(f"{c}_min"),
            spark_max(c).alias(f"{c}_max")
        ]
    print(f"\n   {ds_name.upper()}:")
    display(df.agg(*agg_exprs))

   POST-CLEANING: NULL ANALYSIS
   sales                0 total nulls remaining
   product              0 total nulls remaining
   customer             0 total nulls remaining
   store                0 total nulls remaining
   inventory            0 total nulls remaining
   clickstream          0 total nulls remaining

   POST-CLEANING: DUPLICATE CHECK
   ✅ sales                0 duplicates
   ✅ product              0 duplicates
   ✅ customer             0 duplicates
   ✅ store                0 duplicates
   ✅ inventory            0 duplicates
   ✅ clickstream          0 duplicates

   POST-CLEANING: BUSINESS RULE VALIDATION
   ✅ sales_qty_bad                  0 violations
   ✅ sales_price_bad                0 violations
   ✅ sales_total_bad                0 violations
   ⚠️ product_loss                  10 violations
   ✅ inv_neg_stock                  0 violations

   POST-CLEANING: CATEGORICAL VALUES
   product.category: ['Beauty', 'Clothing', 'Electronics', 'Home', 'Sports']
   pro

quantity_mean,quantity_min,quantity_max,unit_price_mean,unit_price_min,unit_price_max,discount_mean,discount_min,discount_max,net_sales_mean,net_sales_min,net_sales_max,gross_margin_mean,gross_margin_min,gross_margin_max
3.5,1,8,575.21,40.18,1089.3010802094873,0.15,0.0,0.3,1712.55,28.13,8638.54,223.08,-1684.54,2380.12



   PRODUCT:


cost_price_mean,cost_price_min,cost_price_max,selling_price_mean,selling_price_min,selling_price_max
428.8,50.18,799.52,581.83,40.18,1089.3010802094873



   CUSTOMER:


age_mean,age_min,age_max
43.45,18.0,69.0



   INVENTORY:


stock_on_hand_mean,stock_on_hand_min,stock_on_hand_max
245.68,0,500


# 📊 STEP 4: Before vs After Comparison
> Quantify the exact data quality improvement from Bronze to Silver.

In [0]:
# ══════════════════════════════════════════════════════════════════════
# STEP 4: BEFORE vs AFTER COMPARISON
# ══════════════════════════════════════════════════════════════════════
from pyspark.sql import Row

post_row_counts = {name: df.count() for name, df in silver_datasets.items()}

# ── 4a. Row Count Comparison ─────────────────────────────────────────────
print("=" * 70)
print("   4a. ROW COUNT COMPARISON (Bronze → Silver)")
print("=" * 70)
row_rows = [
    Row(
        dataset=name,
        bronze_rows=pre_row_counts[name],
        silver_rows=post_row_counts[name],
        rows_removed=pre_row_counts[name] - post_row_counts[name],
        pct_retained=round(post_row_counts[name] / pre_row_counts[name] * 100, 1)
    )
    for name in pre_row_counts
]
row_comparison_df = spark.createDataFrame(row_rows)
display(row_comparison_df)
print(f"\n   Rejected sales (separate table): {reject_sales.count():,}")

# ── 4b. Null Reduction Per Column ───────────────────────────────────────
print(f"\n{'=' * 70}")
print("   4b. NULL REDUCTION (columns with nulls before cleaning)")
print("=" * 70)
null_rows = []
for name in pre_nulls:
    for col_name, before_count in pre_nulls[name].items():
        if before_count > 0:
            after_count = post_nulls.get(name, {}).get(col_name, 0)
            reduction = round((1 - after_count / before_count) * 100, 1) if before_count > 0 else 0
            null_rows.append(Row(
                dataset=name, column=col_name,
                nulls_before=before_count, nulls_after=after_count,
                pct_reduction=reduction
            ))
if null_rows:
    display(spark.createDataFrame(null_rows))

# ── 4c. Data Quality Improvements ──────────────────────────────────────
print(f"\n{'=' * 70}")
print("   4c. DATA QUALITY IMPROVEMENTS")
print("=" * 70)
quality_rows = [
    Row(rule=rule, before=pre_quality[rule], after=post_quality[rule],
        fixed=pre_quality[rule] - post_quality[rule])
    for rule in pre_quality
]
display(spark.createDataFrame(quality_rows))

# ── 4d. Duplicate Reduction ─────────────────────────────────────────────
print(f"\n{'=' * 70}")
print("   4d. DUPLICATE REDUCTION")
print("=" * 70)
dup_rows = [
    Row(dataset=name, dups_before=pre_duplicates[name], dups_after=post_dups[name],
        removed=pre_duplicates[name] - post_dups[name])
    for name in pre_duplicates
]
display(spark.createDataFrame(dup_rows))

# ── 4e. OVERALL IMPROVEMENT SUMMARY ─────────────────────────────────────
total_pre_nulls  = sum(sum(v.values()) for v in pre_nulls.values())
total_post_nulls = sum(sum(v.values()) for v in post_nulls.values())
total_pre_dups   = sum(pre_duplicates.values())
total_post_dups  = sum(post_dups.values())
total_pre_qual   = sum(pre_quality.values())
total_post_qual  = sum(post_quality.values())

print(f"\n{'=' * 70}")
print("   🏆 OVERALL IMPROVEMENT SUMMARY")
print(f"{'=' * 70}")
print(f"   Total null values:      {total_pre_nulls:>8} → {total_post_nulls:>8}  "
      f"({round((1 - total_post_nulls / max(total_pre_nulls,1)) * 100, 1)}% reduction)")
print(f"   Total duplicates:       {total_pre_dups:>8} → {total_post_dups:>8}  "
      f"({round((1 - total_post_dups / max(total_pre_dups,1)) * 100, 1)}% reduction)")
print(f"   Quality violations:     {total_pre_qual:>8} → {total_post_qual:>8}  "
      f"({round((1 - total_post_qual / max(total_pre_qual,1)) * 100, 1)}% reduction)")
total_bronze = sum(pre_row_counts.values())
total_silver = sum(post_row_counts.values())
print(f"   Total records:          {total_bronze:>8} → {total_silver:>8}  "
      f"({round(total_silver / total_bronze * 100, 1)}% retained)")

   4a. ROW COUNT COMPARISON (Bronze → Silver)


dataset,bronze_rows,silver_rows,rows_removed,pct_retained
sales,1000500,999900,600,99.9
product,500,500,0,100.0
customer,100000,100000,0,100.0
store,100,100,0,100.0
inventory,2500,2500,0,100.0
clickstream,1500000,1500000,0,100.0



   Rejected sales (separate table): 100

   4b. NULL REDUCTION (columns with nulls before cleaning)


dataset,column,nulls_before,nulls_after,pct_reduction
customer,age,200,0,100.0



   4c. DATA QUALITY IMPROVEMENTS


rule,before,after,fixed
sales_qty_bad,50,0,50
sales_price_bad,50,0,50
sales_total_bad,0,0,0
product_loss,10,10,0
inv_neg_stock,20,0,20



   4d. DUPLICATE REDUCTION


dataset,dups_before,dups_after,removed
sales,500,0,500
product,0,0,0
customer,0,0,0
store,0,0,0
inventory,0,0,0
clickstream,0,0,0



   🏆 OVERALL IMPROVEMENT SUMMARY
   Total null values:           200 →        0  (100.0% reduction)
   Total duplicates:            500 →        0  (100.0% reduction)
   Quality violations:          130 →       10  (92.3% reduction)
   Total records:           2603600 →  2603000  (100.0% retained)


# 💾 STEP 5: Output — Write Silver Tables
> Persist all cleaned DataFrames to `medallion.silver` as Delta tables.

In [0]:
# ── Write all 6 cleaned DataFrames to medallion.silver ────────────────────────
silver_sales.write.format("delta").mode("overwrite").saveAsTable("medallion.silver.sales_transactions")
silver_customer.write.format("delta").mode("overwrite").saveAsTable("medallion.silver.customer_data")
silver_product.write.format("delta").mode("overwrite").saveAsTable("medallion.silver.product_master")
silver_store.write.format("delta").mode("overwrite").saveAsTable("medallion.silver.store_master")
silver_inventory.write.format("delta").mode("overwrite").saveAsTable("medallion.silver.inventory_data")
silver_clickstream.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("medallion.silver.clickstream_events")

print("All 6 tables written to medallion.silver successfully.")

All 6 tables written to medallion.silver successfully.


In [0]:
# ─── DERIVED FIELDS: net_sales, gross_margin, order_value_band ────────────────
from pyspark.sql.functions import col, when, round as spark_round, lit, current_timestamp

print("Adding derived fields to Silver sales...")

# First check if cost_price already exists in silver_sales
# If it does, no need to join again
if "cost_price" in silver_sales.columns:
    # cost_price already in the table — use it directly
    silver_sales_enriched = silver_sales \
        .withColumn(
            "net_sales",
            spark_round(col("total_amount"), 2)
        ) \
        .withColumn(
            "gross_margin",
            spark_round(
                col("total_amount") - (col("quantity") * col("cost_price")),
                2
            )
        ) \
        .withColumn(
            "order_value_band",
            when(col("net_sales") < 500,  "Low")
           .when(col("net_sales") < 2000, "Medium")
           .otherwise("High")
        )
else:
    # cost_price not in silver_sales — need to join from product master
    silver_sales_enriched = silver_sales \
        .join(
            silver_product.select("product_id", "cost_price").distinct(),
            on="product_id",
            how="left"
        ) \
        .withColumn(
            "net_sales",
            spark_round(col("total_amount"), 2)
        ) \
        .withColumn(
            "gross_margin",
            spark_round(
                col("total_amount") - (col("quantity") * col("cost_price")),
                2
            )
        ) \
        .withColumn(
            "order_value_band",
            when(col("net_sales") < 500,  "Low")
           .when(col("net_sales") < 2000, "Medium")
           .otherwise("High")
        )

print("✅ Derived fields added. Sample:")
silver_sales_enriched.select(
    "transaction_id", "net_sales", "gross_margin", "order_value_band"
).show(5)

Adding derived fields to Silver sales...
✅ Derived fields added. Sample:
+--------------+---------+------------+----------------+
|transaction_id|net_sales|gross_margin|order_value_band|
+--------------+---------+------------+----------------+
|       5004036|  2108.26|       73.15|            High|
|       5009107|  3864.06|      298.31|            High|
|       5014484|   596.99|      181.47|          Medium|
|       5025426|   388.78|      292.64|             Low|
|       5030404|  1819.59|       454.8|          Medium|
+--------------+---------+------------+----------------+
only showing top 5 rows


In [0]:
# ─── WRITE ENRICHED SALES TO SILVER ──────────────────────────────────────────
print("Writing enriched sales to Silver layer...")

silver_sales_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_date", "channel") \
    .saveAsTable("medallion.silver.sales_transactions")

print("✅ medallion.silver.sales_transactions written with derived fields")

Writing enriched sales to Silver layer...
✅ medallion.silver.sales_transactions written with derived fields


In [0]:
# ─── REJECT LOG: capture bad records with reason ─────────────────────────────
print("Running data quality checks and building reject log...")

silver_sales     = spark.table("medallion.silver.sales_transactions")
silver_inventory = spark.table("medallion.silver.inventory_data")
silver_product   = spark.table("medallion.silver.product_master")

from pyspark.sql.functions import col, lit, current_timestamp

rules = [
    ("quantity <= 0",         silver_sales.filter(col("quantity") <= 0)),
    ("unit_price <= 0",       silver_sales.filter(col("unit_price") <= 0)),
    ("total_amount < 0",      silver_sales.filter(col("total_amount") < 0)),
    ("gross_margin < 0",      silver_sales.filter(col("gross_margin") < 0)),
    ("stock_on_hand < 0",     silver_inventory.filter(col("stock_on_hand") < 0)),
    ("selling < cost_price",  silver_product.filter(
                                  col("selling_price") < col("cost_price"))),
]

for rule_name, bad_df in rules:
    count = bad_df.count()

    log_df = spark.createDataFrame([{
        "rule":          rule_name,
        "violation_count": count,
        "status":        "PASS" if count == 0 else "FAIL"
    }]).withColumn("checked_at", current_timestamp())

    log_df.write.format("delta").mode("append") \
       .option("mergeSchema", "true") \
       .saveAsTable("medallion.silver.dq_rule_log")

    if count > 0:
        bad_df.withColumn("reject_reason", lit(rule_name)) \
              .withColumn("rejected_at",   current_timestamp()) \
              .write.format("delta").mode("append") \
              .option("mergeSchema", "true") \
              .saveAsTable("medallion.silver.dq_reject_table")

        print(f"  ⚠️  Rule FAILED: {rule_name}  →  {count} bad records saved to reject table")
    else:
        print(f"  ✅ Rule PASSED: {rule_name}")

print("\nDone. Reject log summary:")
display(spark.table("medallion.silver.dq_rule_log"))

Running data quality checks and building reject log...
  ✅ Rule PASSED: quantity <= 0
  ✅ Rule PASSED: unit_price <= 0
  ✅ Rule PASSED: total_amount < 0
  ⚠️  Rule FAILED: gross_margin < 0  →  30558 bad records saved to reject table
  ✅ Rule PASSED: stock_on_hand < 0
  ⚠️  Rule FAILED: selling < cost_price  →  10 bad records saved to reject table

Done. Reject log summary:


rule,status,violation_count,checked_at
quantity <= 0,PASS,0,2026-04-03T06:20:46.154556Z
unit_price <= 0,PASS,0,2026-04-03T06:20:47.537328Z
total_amount < 0,PASS,0,2026-04-03T06:20:48.662128Z
gross_margin < 0,FAIL,266507,2026-04-03T06:20:58.316875Z
stock_on_hand < 0,PASS,0,2026-04-03T06:21:08.986814Z
selling < cost_price,FAIL,10,2026-04-03T06:21:10.369669Z
quantity <= 0,PASS,0,2026-04-03T06:40:59.546598Z
unit_price <= 0,PASS,0,2026-04-03T06:41:00.933734Z
total_amount < 0,PASS,0,2026-04-03T06:41:01.91291Z
gross_margin < 0,FAIL,30558,2026-04-03T06:41:10.081344Z


In [0]:
display(
    spark.table("medallion.silver.sales_transactions")
    .select("transaction_id", "product_id", "quantity", "unit_price", 
            "discount", "net_sales", "gross_margin", "order_value_band")
    .limit(100)
)

transaction_id,product_id,quantity,unit_price,discount,net_sales,gross_margin,order_value_band
5015627,1020,3,298.78688969106366,0.01,896.35,620.11,Medium
5019569,1287,3,603.2926297103495,0.18,1809.7,-275.11,Medium
5020069,1191,3,469.5843803423065,0.16,1408.59,35.65,Medium
5022101,1304,1,798.1696477933551,0.11,798.06,185.11,Medium
5029312,1455,1,780.405400909965,0.25,780.16,136.52,Medium
5048192,1006,4,718.3855570958937,0.19,2873.35,145.46,High
5052031,1007,1,333.3901572687763,0.29,333.1,216.13,Low
5054611,1245,4,161.50011057521266,0.01,645.99,440.26,Medium
5054875,1431,3,865.0502411841719,0.26,2594.89,-465.01,High
5062496,1491,1,524.4987930820533,0.23,524.27,-27.85,Medium


In [0]:
%sql
OPTIMIZE medallion.silver.sales_transactions
  ZORDER BY (product_id, customer_id, store_id);

OPTIMIZE medallion.silver.product_master  ZORDER BY (product_id);
OPTIMIZE medallion.silver.customer_data   ZORDER BY (customer_id);
OPTIMIZE medallion.silver.store_master    ZORDER BY (store_id);

SELECT 'Silver OPTIMIZE complete' AS status;

status
Silver OPTIMIZE complete


In [0]:
%sql
-- This query confirms that each city is mapped to exactly ONE correct state
SELECT city, state, COUNT(*) as store_count
FROM medallion.silver.store_master
GROUP BY city, state
ORDER BY city;

city,state,store_count
Bangalore,Karnataka,11
Chandigarh,Chandigarh,10
Delhi,Delhi,17
Hyderabad,Telangana,10
Jaipur,Rajasthan,15
Mumbai,Maharashtra,22
Pune,Maharashtra,15
